In [1]:
import pandas as pd

model2_df = pd.read_parquet("model2_dataset.parquet")

print("Rows:", len(model2_df))

print("\nColumns:")
print(model2_df.columns.tolist())

print("\nTarget statistics:")
print(model2_df["dbt_target"].describe())

print("\nMissing values:")
print(model2_df.isnull().sum())

Rows: 197

Columns:
['invoice_id', 'buyer_id', 'issue_date', 'buyer_dbt_mean', 'buyer_dbt_sd', 'buyer_dispute_rate', 'buyer_promise_keep', 'amount_rel', 'terms', 'quarter_end', 'late_target', 'dbt_target']

Target statistics:
count    197.000000
mean      15.685279
std        9.118407
min        1.000000
25%       10.000000
50%       15.000000
75%       21.000000
max       57.000000
Name: dbt_target, dtype: float64

Missing values:
invoice_id            0
buyer_id              0
issue_date            0
buyer_dbt_mean        0
buyer_dbt_sd          0
buyer_dispute_rate    0
buyer_promise_keep    0
amount_rel            0
terms                 0
quarter_end           0
late_target           0
dbt_target            0
dtype: int64


In [2]:
# Make sure dates are proper datetime values
model2_df["issue_date"] = pd.to_datetime(model2_df["issue_date"])

# Sort chronologically
model2_df = model2_df.sort_values("issue_date").reset_index(drop=True)

# Use the same test-period boundary as Model 1
test_start = pd.Timestamp("2026-03-19")

train_m2 = model2_df[model2_df["issue_date"] < test_start].copy()
test_m2 = model2_df[model2_df["issue_date"] >= test_start].copy()

print("===== MODEL 2 TEMPORAL SPLIT =====")

print("\nTRAINING SET")
print("Rows:", len(train_m2))
print("Date range:",
      train_m2["issue_date"].min(),
      "to",
      train_m2["issue_date"].max())

print("\nTEST SET")
print("Rows:", len(test_m2))
print("Date range:",
      test_m2["issue_date"].min(),
      "to",
      test_m2["issue_date"].max())

print("\nTarget distribution")
print("Training:")
print(train_m2["dbt_target"].describe())

print("\nTest:")
print(test_m2["dbt_target"].describe())

===== MODEL 2 TEMPORAL SPLIT =====

TRAINING SET
Rows: 129
Date range: 2026-01-01 00:00:00 to 2026-03-18 00:00:00

TEST SET
Rows: 68
Date range: 2026-03-21 00:00:00 to 2026-04-30 00:00:00

Target distribution
Training:
count    129.000000
mean      15.713178
std        9.165317
min        1.000000
25%        9.000000
50%       15.000000
75%       21.000000
max       57.000000
Name: dbt_target, dtype: float64

Test:
count    68.000000
mean     15.632353
std       9.096341
min       1.000000
25%      10.000000
50%      15.000000
75%      20.250000
max      47.000000
Name: dbt_target, dtype: float64


In [3]:
# Features used by Model 2
features = [
    "buyer_dbt_mean",
    "buyer_dbt_sd",
    "buyer_dispute_rate",
    "buyer_promise_keep",
    "amount_rel",
    "terms",
    "quarter_end"
]

X_train_m2 = train_m2[features]
y_train_m2 = train_m2["dbt_target"]

X_test_m2 = test_m2[features]
y_test_m2 = test_m2["dbt_target"]

print("===== MODEL 2 DATA =====")
print("X_train:", X_train_m2.shape)
print("y_train:", y_train_m2.shape)
print("X_test :", X_test_m2.shape)
print("y_test :", y_test_m2.shape)

print("\nFeatures:")
print(features)

print("\nFirst 5 training rows:")
display(X_train_m2.head())

print("\nFirst 5 targets:")
display(y_train_m2.head())

===== MODEL 2 DATA =====
X_train: (129, 7)
y_train: (129,)
X_test : (68, 7)
y_test : (68,)

Features:
['buyer_dbt_mean', 'buyer_dbt_sd', 'buyer_dispute_rate', 'buyer_promise_keep', 'amount_rel', 'terms', 'quarter_end']

First 5 training rows:


,buyer_dbt_mean,buyer_dbt_sd,buyer_dispute_rate,buyer_promise_keep,amount_rel,terms,quarter_end
0,0.000000,0.000000,0.0,0.5,1.0,60,0
1,8.000000,0.000000,0.0,0.5,1.0,30,0
2,9.333333,14.047538,0.0,0.5,1.0,60,0
3,9.750000,11.500000,0.0,0.5,1.0,30,0
4,11.400000,10.620734,0.0,0.5,1.0,30,0



First 5 targets:


,dbt_target
0,8
1,24
2,11
3,18
4,19


In [4]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Baseline Model 2
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

# Train
gb_model.fit(X_train_m2, y_train_m2)

# Predict on untouched test set
y_pred_m2 = gb_model.predict(X_test_m2)

# Evaluation
mae = mean_absolute_error(y_test_m2, y_pred_m2)
rmse = np.sqrt(mean_squared_error(y_test_m2, y_pred_m2))
r2 = r2_score(y_test_m2, y_pred_m2)

print("===== MODEL 2 BASELINE RESULTS =====")
print(f"MAE  : {mae:.4f} days")
print(f"RMSE : {rmse:.4f} days")
print(f"R²   : {r2:.4f}")

===== MODEL 2 BASELINE RESULTS =====
MAE  : 6.3048 days
RMSE : 8.0542 days
R²   : 0.2043


In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

gb_model.fit(X_train_m2, y_train_m2)

y_pred_m2 = gb_model.predict(X_test_m2)

mae = mean_absolute_error(y_test_m2, y_pred_m2)
rmse = np.sqrt(mean_squared_error(y_test_m2, y_pred_m2))
r2 = r2_score(y_test_m2, y_pred_m2)

print("===== MODEL 2 BASELINE RESULTS =====")
print(f"MAE  : {mae:.4f} days")
print(f"RMSE : {rmse:.4f} days")
print(f"R²   : {r2:.4f}")

===== MODEL 2 BASELINE RESULTS =====
MAE  : 6.3048 days
RMSE : 8.0542 days
R²   : 0.2043


In [6]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np
import pandas as pd

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

param_grid = [
    {"n_estimators": 50,  "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 100, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.03, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.03, "max_depth": 2},

    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},

    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},

    {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 2},
    {"n_estimators": 150, "learning_rate": 0.10, "max_depth": 2},
    {"n_estimators": 200, "learning_rate": 0.10, "max_depth": 2},
]

results = []

for params in param_grid:

    model = GradientBoostingRegressor(
        **params,
        random_state=42
    )

    # Negative MAE because sklearn maximizes scores
    scores = cross_val_score(
        model,
        X_train_m2,
        y_train_m2,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    results.append({
        **params,
        "Mean MAE": -scores.mean(),
        "Std MAE": scores.std()
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Mean MAE"
).reset_index(drop=True)

display(results_df)

,n_estimators,learning_rate,max_depth,Mean MAE,Std MAE
0,100,0.03,2,5.923266,0.712753
1,150,0.03,2,5.971836,0.622774
2,100,0.05,2,6.024522,0.614840
3,200,0.03,2,6.060165,0.567609
4,50,0.03,2,6.105626,0.703556
5,150,0.05,2,6.184891,0.599893
6,100,0.05,3,6.250567,0.588259
7,100,0.10,2,6.271599,0.503581
8,200,0.05,2,6.321060,0.500765
9,150,0.05,3,6.401751,0.613405


In [7]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

final_m2 = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.03,
    max_depth=2,
    random_state=42
)

# Train on the complete training set
final_m2.fit(X_train_m2, y_train_m2)

# Predict the unseen test set
y_pred_final = final_m2.predict(X_test_m2)

# Evaluate
final_mae = mean_absolute_error(y_test_m2, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test_m2, y_pred_final))
final_r2 = r2_score(y_test_m2, y_pred_final)

print("===== MODEL 2 FINAL RESULTS =====")
print(f"MAE  : {final_mae:.4f} days")
print(f"RMSE : {final_rmse:.4f} days")
print(f"R²   : {final_r2:.4f}")

===== MODEL 2 FINAL RESULTS =====
MAE  : 6.0255 days
RMSE : 7.5655 days
R²   : 0.2979


In [8]:
# Create an error-analysis table

error_df = test_m2[
    ["invoice_id", "buyer_id", "issue_date", "dbt_target"]
].copy()

error_df["predicted_dbt"] = y_pred_final

# Prediction error
error_df["error"] = (
    error_df["predicted_dbt"] - error_df["dbt_target"]
)

# Absolute error
error_df["absolute_error"] = (
    error_df["error"].abs()
)

# Sort by largest mistakes
error_df = error_df.sort_values(
    "absolute_error",
    ascending=False
).reset_index(drop=True)

print("===== MODEL 2 ERROR ANALYSIS =====")

print("\nLargest prediction errors:")
display(error_df.head(10))

===== MODEL 2 ERROR ANALYSIS =====

Largest prediction errors:


,invoice_id,buyer_id,issue_date,dbt_target,predicted_dbt,error,absolute_error
0,INV-2028,BUY-01,2026-04-22,12,34.411485,22.411485,22.411485
1,INV-2181,BUY-10,2026-04-28,33,17.471381,-15.528619,15.528619
2,INV-2037,BUY-01,2026-04-24,19,34.411485,15.411485,15.411485
3,INV-2044,BUY-01,2026-03-30,19,34.411485,15.411485,15.411485
4,INV-2175,BUY-10,2026-04-26,5,17.786212,12.786212,12.786212
5,INV-2045,BUY-01,2026-04-06,47,34.411485,-12.588515,12.588515
6,INV-2208,BUY-14,2026-03-30,20,7.433695,-12.566305,12.566305
7,INV-2146,BUY-07,2026-04-13,30,17.480826,-12.519174,12.519174
8,INV-2145,BUY-07,2026-04-29,30,17.863146,-12.136854,12.136854
9,INV-2137,BUY-07,2026-04-29,31,18.870228,-12.129772,12.129772


In [9]:
print("===== ERROR SUMMARY =====")

print("\nMean Error (Bias):")
print(error_df["error"].mean())

print("\nMean Absolute Error:")
print(error_df["absolute_error"].mean())

print("\nMedian Absolute Error:")
print(error_df["absolute_error"].median())

print("\nMaximum Absolute Error:")
print(error_df["absolute_error"].max())

print("\nPredictions within ±5 days:")
within_5 = (error_df["absolute_error"] <= 5).mean() * 100
print(f"{within_5:.2f}%")

print("\nPredictions within ±10 days:")
within_10 = (error_df["absolute_error"] <= 10).mean() * 100
print(f"{within_10:.2f}%")

===== ERROR SUMMARY =====

Mean Error (Bias):
0.6546371932044283

Mean Absolute Error:
6.025509715544151

Median Absolute Error:
5.262168013527523

Maximum Absolute Error:
22.411485007832468

Predictions within ±5 days:
45.59%

Predictions within ±10 days:
80.88%


In [10]:
import pandas as pd

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": final_m2.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("===== MODEL 2 FEATURE IMPORTANCE =====")
display(importance_df)

===== MODEL 2 FEATURE IMPORTANCE =====


,Feature,Importance
0,buyer_dbt_mean,0.678776
1,amount_rel,0.147428
2,buyer_dbt_sd,0.125538
3,quarter_end,0.036948
4,buyer_promise_keep,0.007207
5,terms,0.004103
6,buyer_dispute_rate,0.000000


In [11]:
import joblib

joblib.dump(final_m2, "model2.joblib")

print("Model 2 saved successfully as model2.joblib")

Model 2 saved successfully as model2.joblib
